# Fisher Information Geometry: Metric Evolution with Depth

Shows how the Fisher metric contracts layer by layer. Computes eigenvalue spectrum at each depth.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Fisher Metric Pullback Across Depth

In [ ]:
from src.core.fisher.fisher_metric import FisherMetric
from src.core.fisher.eigenvalue_analyzer import FisherEigenvalueAnalyzer
import torch

fm = FisherMetric()
analyzer = FisherEigenvalueAnalyzer()
n = 32
G = torch.eye(n)
depths = [0, 5, 10, 20, 50]
metrics_at_depths = {0: G.clone()}

sigma_w = 1.4
for ell in range(1, max(depths)+1):
    J = torch.randn(n, n) * (sigma_w / n**0.5)
    G = fm.pullback(G, J)
    if ell in depths:
        metrics_at_depths[ell] = G.clone()

fig, ax = plt.subplots(figsize=(8, 5))
for d, G_d in metrics_at_depths.items():
    result = analyzer.analyze(G_d)
    ev = np.sort(result.eigenvalues)[::-1]
    ax.semilogy(range(len(ev)), ev, label=f"depth={d}")
ax.set_xlabel("Eigenvalue index"); ax.set_ylabel("Eigenvalue (log scale)")
ax.set_title("Fisher metric eigenvalue spectrum contracts with depth")
ax.legend(); plt.tight_layout(); plt.show()

# Print effective dimensions
print("\nEffective dimension at each depth:")
for d, G_d in metrics_at_depths.items():
    r = analyzer.analyze(G_d)
    print(f"  depth={d:3d}: d_eff={r.effective_dimension:.1f}, kappa={r.condition_number:.2e}")
